# Context-source ablations - report

Reads the newest run under `results/`, or the one named by
`JUMPER_ABLATION_RESULTS`. Run top to bottom.

It reports; it does not measure. Everything here comes from
`summary.csv`, which comes from `metrics.csv`, which comes from the
records - so any number can be chased back to the reviewer
invocation that produced it.

Produce those first:

```bash
python -m jumper_ablations.cli.run suite=context_sources
python -m jumper_ablations.cli.export_judge   # then judge them
python -m jumper_ablations.cli.evaluate
python -m jumper_ablations.cli.report
```

## 0. Load the run

In [ ]:
import os
from pathlib import Path

import pandas as pd

from jumper_ablations.runner.run_directory import RunDirectory
from jumper_ablations.paths import RESULTS_DIR

named = os.environ.get("JUMPER_ABLATION_RESULTS")
run = (
    RunDirectory(Path(named))
    if named
    else RunDirectory.latest(RESULTS_DIR)
)

summary = pd.read_csv(run.path / "summary.csv")
usecases = sorted(summary["usecase"].unique())
print(run.path)
print(f"{len(usecases)} usecase(s), "
      f"{summary['ablation'].nunique()} preset(s), "
      f"{len(summary)} summary row(s)")

## 1. Did every pass actually run?

The first thing to check, and the one most likely to invalidate
everything below. A pass that failed in the prefix never reached the
reviewer; a record with no populated sources means the model was
asked to analyse nothing and answered anyway.

In [ ]:
passes = pd.read_json(run.passes_index, lines=True)
display(passes)

failed = passes[passes["status"] != "ok"]
if len(failed):
    print(f"!! {len(failed)} pass(es) did not complete")

runs = pd.read_csv(run.path / "runs.csv")
degraded = runs[runs["degraded"].fillna(False)]
if len(degraded):
    print(
        f"!! {len(degraded)} record(s) fell back to the full replay; "
        "their replay mode is not the one that was requested"
    )

## 2. How much was judged?

Every analysis metric is judged by hand. A metric averaged over the
third of units that happened to get judged is not a measurement, so
the gaps are read before the numbers.

In [ ]:
missing_path = run.judge / "missing.csv"
if missing_path.is_file():
    missing = pd.read_csv(missing_path)
    display(
        missing.groupby(["metric", "reason"])
        .size()
        .rename("units")
        .reset_index()
    )
else:
    print("no gaps recorded - either everything was judged, "
          "or evaluate has not run")

## 3. The two tables

In [ ]:
from IPython.display import Markdown

for name in ("analysis_metrics.md", "suggestions_metrics.md"):
    display(Markdown((run.path / name).read_text()))

## 4. Figures

Hover for exact values. Set `MODE = "dark"` if the notebook theme
is dark - the palette has its own dark steps rather than an
automatic flip.

In [ ]:
from jumper_ablations.report import (
    correctness_rates,
    evidence_coverage,
    paired_deltas,
    speedup_with_coverage,
    write_figures,
)

MODE = "light"
BASELINE = "base"

figures = {}
for usecase in usecases:
    slug = usecase.replace("/", "-")
    figures[f"{slug}__speedup"] = speedup_with_coverage(
        summary, usecase, BASELINE, MODE
    )
    figures[f"{slug}__correctness"] = correctness_rates(
        summary, usecase, BASELINE, MODE
    )
    figures[f"{slug}__coverage"] = evidence_coverage(
        summary, usecase, BASELINE, MODE
    )
    figures[f"{slug}__delta_speedup"] = paired_deltas(
        summary,
        usecase,
        "correctness_conditioned_speedup",
        "geometric_mean_speedup",
        BASELINE,
        MODE,
    )

for figure in figures.values():
    if figure is not None:
        figure.show()

### PNG export

Optional, and it needs `kaleido>=1` for plotly 6+. The extension
itself pins `kaleido==0.2.1` for its own plotly 5 charts, so the two
cannot both be satisfied in one environment - the interactive
figures above do not need it either way.

In [ ]:
print(write_figures(figures, run.path / "figures"))

## 5. Reading it

Three pairings, and none of the halves means anything alone:

- **speedup and coverage.** 3.0x over 10% of suggestions and 1.8x
  over 90% are not close, however the first one looks.
- **conditional and global evidence coverage.** High conditional and
  low global is the finding this experiment is built to see: the
  model used its context well, and the context was not enough.
- **any difference and its interval.** A paired delta whose interval
  crosses zero is not a result. If most of them do, the answer is
  more generations, not a bolder claim.